In [3]:
from dotenv import load_dotenv

load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "What is psychotherapy? Answer in one sentence"}
    ],
)

In [ ]:
block = message.content[0]
if block.type == "text":
    print(block.text)

Psychotherapy is a treatment process where a trained mental health professional helps people address psychological difficulties, emotional distress, or behavioral problems through structured conversations and evidence-based techniques.


In [4]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages):
    system_prompt = """
    You are a patient math tutor.
    Do not directly answer a student's questions.
    Guide them to a solution step by step.
    """
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        system=system_prompt
    )
    block = message.content[0]
    if block.type == "text":
        return block.text

In [12]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "What is psychotherapy? Answer in one sentence")

# Get Claude's response
answer = chat(messages)
print(answer)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)
print(final_answer)

Psychotherapy is a treatment process in which a trained mental health professional helps a person address psychological issues, emotional distress, or behavioral problems through conversation and evidence-based techniques.
It typically involves exploring thoughts, feelings, and behaviors in a confidential, supportive environment to promote healing, personal growth, and improved mental health.


In [ ]:
messages = []

while True:
    # get user input
    user_input = input("You: ")

    # exit gracefully on empty input (e.g. pressing Esc) or explicit quit command
    if not user_input.strip() or user_input.strip().lower() in ("quit", "exit"):
        print("Ending chat. Goodbye!")
        break

    print(f"User input: {user_input}")

    # add user message to the conversation history
    add_user_message(messages, user_input)

    # call claude with the chat function
    response = chat(messages)

    # print the response
    print(f"Claude: {response}")

    # add generated response to the conversation history
    add_assistant_message(messages, response)

In [34]:
# flexible system prompt
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# example
messages = []

add_user_message(messages, "Generate a youtube video idea. Channel focus is software, AI, psychology of tech")

answer = chat(messages, system=None)

answer


'# YouTube Video Idea: "Why Apps Hijack Your Brain: The Dark Patterns Hidden in Your Phone"\n\n## Concept\nA video exposing the psychological manipulation tactics built into popular apps, combining software engineering breakdown with behavioral psychology insights.\n\n## Structure\n\n**Hook (0-30s)**\n- Show yourself using an app normally, then freeze frame on addictive UI elements\n- "These buttons are designed by psychologists to hijack your attention"\n\n**Part 1: The Tech (3-4 min)**\n- Code walkthrough: notification systems, variable rewards, infinite scroll algorithms\n- Show actual app source code or architecture explaining:\n  - How algorithms calculate "optimal" notification timing\n  - Streak mechanics (Snapchat, Duolingo)\n  - Red badge icon triggers\n\n**Part 2: The Psychology (4-5 min)**\n- Interview or animated explainer featuring:\n  - Variable ratio reward schedules (Skinner\'s operant conditioning)\n  - FOMO engineering\n  - Loss aversion in streaks\n  - Social proof m

In [ ]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

In [21]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

# FakeDB

FakeDB is a lightweight, in-memory database that generates and stores randomized mock data for application testing, allowing developers to quickly populate their systems with realistic but fictional records without requiring external data sources.

In [ ]:
# A chat returning just code blocks without any extra text or commentary
import re

messages = []

prompt = """
Give me examples of Node.js functions for a fake database that can be used to store and retrieve user data.
"""

add_user_message(messages, prompt)
prefill = "```javascript"
add_assistant_message(messages, prefill)

text = chat(messages, stop_sequences=["```"])
code = re.sub(r"^[ \t]*//.*\n?", "", text, flags=re.MULTILINE)
code = re.sub(r"\n{3,}", "\n\n", code).strip()
full_text = f"{prefill}\n{code}\n```"

from IPython.display import Markdown
Markdown(full_text)


```javascript
const users = {};
let userIdCounter = 1;

function createUser(userData) {
  const userId = userIdCounter++;
  const user = {
    id: userId,
    ...userData,
    createdAt: new Date(),
  };
  users[userId] = user;
  return user;
}

function getUserById(userId) {
  return users[userId] || null;
}

function getAllUsers() {
  return Object.values(users);
}

function getUserByEmail(email) {
  return Object.values(users).find(user => user.email === email) || null;
}

function updateUser(userId, updates) {
  if (!users[userId]) {
    return null;
  }
  users[userId] = {
    ...users[userId],
    ...updates,
    updatedAt: new Date(),
  };
  return users[userId];
}

function deleteUser(userId) {
  if (users[userId]) {
    delete users[userId];
    return true;
  }
  return false;
}

function userExists(userId) {
  return userId in users;
}

function getUserCount() {
  return Object.keys(users).length;
}

module.exports = {
  createUser,
  getUserById,
  getAllUsers,
  getUserByEmail,
  updateUser,
  deleteUser,
  userExists,
  getUserCount,
};
```